# Real Speech Adaptation and Speaker Verification
Two reproducible experiments for a speech-ML portfolio: Whisper decoder LoRA and WavLM speaker verification. This notebook calls the tested repository modules; it does not duplicate training logic.

The default pilot uses 128 real training recordings, 48 validation recordings, and 48 test recordings from disjoint speakers. English audiobook speech only. Full experiment outputs, including unsuccessful comparisons, are saved. No accounts or API keys are needed. A GPU accelerates the run; the published pilot was run on CPU.

Choose **Runtime → Change runtime type → T4 GPU** on Colab if available. Downloads include pretrained checkpoints and several LibriSpeech Parquet shards (roughly 1–2 GB for the pilot); allow several GB of free disk for encoder caches. Do not treat this as a full-corpus benchmark or an assistant-intent detector.

In [ ]:
import pathlib, subprocess, sys
REPO = pathlib.Path.cwd() if (pathlib.Path.cwd() / 'speechlab').exists() else (pathlib.Path('/content/Speech-Command-') if pathlib.Path('/content').exists() else pathlib.Path.cwd() / 'Speech-Command-')
BRANCH = 'codex/siri-real-speech-lab-20261119'
if not (REPO / '.git').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                           'https://github.com/ByteBoss123/Speech-Command-.git', str(REPO)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-speech.txt')])
print(REPO)

If this runtime had already imported a different Torch or Transformers version, restart the runtime after installation and rerun the setup cell. All following commands fail loudly if an experiment fails; results are not fabricated or substituted.

In [ ]:
import datetime, json
RUN = 'runs/notebook-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
def run(*args):
    subprocess.check_call([sys.executable, *args], cwd=REPO)
run('-m', 'pytest', 'tests_speech', '-q')
DATA = 'data/librispeech-pilot'
if not (REPO / DATA / 'manifest.jsonl').exists():
    run('-m', 'speechlab.prepare', '--output', DATA,
        '--revision', '71cacbfb7e2354c4226d01e70d77d5fca3d04ba1')
print(json.loads((REPO / DATA / 'dataset_card.json').read_text()))

## Fine-tune and evaluate Whisper
The encoder is frozen and cached. Only decoder attention LoRA weights train. The best epoch is chosen by validation loss; test transcripts never influence model selection. Baseline and adapted predictions use identical decoding settings. The optional noise test applies fixed 10 dB Gaussian noise to the same recordings.

In [ ]:
run('-m', 'speechlab.asr', '--manifest', DATA + '/manifest.jsonl', '--output', RUN + '/asr',
    '--revision', '169d4a4341b33bc18d8881c4b69c2e104e1cc0af', '--noise-snr', '10')
print((REPO / RUN / 'asr/metrics.json').read_text())

## Speaker verification
WavLM is used as a **frozen pretrained model**, not trained here. The operating threshold is selected using validation speakers only, then applied to different test speakers. Same-speaker trials in this pilot can share an audiobook chapter, so this is easier than cross-session verification. FAR is false-accept rate; FRR is false-reject rate.

In [ ]:
run('-m', 'speechlab.speaker', '--manifest', DATA + '/manifest.jsonl', '--output', RUN + '/speaker',
    '--revision', 'feb593a6c23c1cc3d9510425c29b0a14d2b07b1e')
print((REPO / RUN / 'speaker/metrics.json').read_text())

## Save the results
Keep raw predictions, trial scores, configuration, training history, and adapter weights. The export below excludes the large frozen-encoder cache. Negative results are retained. These artifacts are needed to substantiate any resume numbers.

In [ ]:
import zipfile
archive = REPO / (RUN.replace('/', '_') + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in (REPO / RUN).rglob('*'):
        if path.is_file() and 'encoder_cache' not in path.parts:
            z.write(path, path.relative_to(REPO / RUN))
    for name in ['manifest.jsonl', 'dataset_card.json']:
        z.write(REPO / DATA / name, 'data/' + name)
print(archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass